In [6]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import gc

In [7]:
housing = fetch_california_housing()
X = housing.data
median_price = np.median(housing.target)
y = (housing.target > median_price).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_dataset  = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=30, shuffle=False)

In [8]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.layers(x)

model = mlp()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
gc.collect()

2400

In [9]:
epochs = 20
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [10]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    
    gc.collect()

Epoch [1/20]
Training loss 0.39828154327015775
Training accuracy 83.11531007751938 %
Testing loss 0.38432450341277347
Testing accuracy 84.64147286821705 %
Epoch [2/20]
Training loss 0.3725521384391847
Training accuracy 85.1501937984496 %
Testing loss 0.3436296475228182
Testing accuracy 84.7625968992248 %
Epoch [3/20]
Training loss 0.35148511823743234
Training accuracy 85.41666666666667 %
Testing loss 0.3489412240640715
Testing accuracy 85.1501937984496 %
Epoch [4/20]
Training loss 0.33788293100128225
Training accuracy 85.63468992248062 %
Testing loss 0.3283362035574608
Testing accuracy 86.19186046511628 %
Epoch [5/20]
Training loss 0.3423991388345296
Training accuracy 85.62257751937985 %
Testing loss 0.33382363013143457
Testing accuracy 85.87693798449612 %
Epoch [6/20]
Training loss 0.3355735417080018
Training accuracy 85.7376453488372 %
Testing loss 0.3427377980808879
Testing accuracy 85.44089147286822 %
Epoch [7/20]
Training loss 0.3413068869966985
Training accuracy 85.99200581395348